<a href="https://colab.research.google.com/github/6pb4wnww4g-beep/Jason/blob/main/Jason_Market_Master.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- JASON MARKET MASTER v1 ---
# Formål:
# Samler hele market-kjeden i ett script.
#
# Kjør dette før Jason-hovedscriptet.
#
# Gjør alt dette:
# 1. Henter EPL-odds fra The Odds API
# 2. Skriver rå/ryddet OddsAPI-data
# 3. Beregner Market_xG_All_Fixtures_v1
# 4. Beregner Market_Team_Strength_v2_1
# 5. Beregner Market_Match_Strength_v2_1
#
# Skriver til Google Sheets:
# - OddsAPI_Raw_Matches
# - OddsAPI_Market_Check
# - OddsAPI_Bookmaker_Detail
# - Market_xG_All_Fixtures_v1
# - Market_xG_Parse_Check_v1
# - Market_Decimal_Check_v2_1
# - Market_Team_Strength_v2_1
# - Market_Match_Strength_v2_1
#
# Viktig:
# - Sett API_KEY før kjøring.
# - Scriptet henter alle EPL-kamper The Odds API returnerer/priser nå.
# - Ingen hardkodet GW1-5-liste.
# - Jason-hovedscriptet leser Market_Match_Strength_v2_1.

import time
import math
import requests
import numpy as np
import pandas as pd
from google.colab import auth
import gspread
from google.auth import default


# -----------------------
# KONFIGURASJON
# -----------------------

API_KEY = "1febfae459fe024b33c82de75598bd9f"

SPREADSHEET_NAME = "Jason Development"

SPORT_KEY = "soccer_epl"
REGIONS = "uk"
MARKETS = "h2h,totals"
ODDS_FORMAT = "decimal"
DATE_FORMAT = "iso"
BASE_URL = "https://api.the-odds-api.com/v4"

DEFAULT_TOTAL_GOALS = 2.65

MIN_SCALE = 10
MAX_SCALE = 95


# -----------------------
# Google Sheets
# -----------------------

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open(SPREADSHEET_NAME)


# -----------------------
# Generelle helpers
# -----------------------

def update_worksheet(sheet_name, dataframe, min_rows=200, min_cols=30, sleep_seconds=2):
    print(f"Oppdaterer {sheet_name}...")
    df_clean = dataframe.replace([np.inf, -np.inf], np.nan).fillna("")

    try:
        ws = sh.worksheet(sheet_name)
        ws.clear()
    except Exception:
        ws = sh.add_worksheet(title=sheet_name, rows=str(min_rows), cols=str(min_cols))

    ws.resize(
        rows=max(min_rows, len(df_clean) + 20),
        cols=max(min_cols, len(df_clean.columns) + 5)
    )

    ws.update([df_clean.columns.tolist()] + df_clean.values.tolist())
    time.sleep(sleep_seconds)


def parse_float(v, default=np.nan):
    if v is None:
        return default

    s = str(v).strip()
    if s == "":
        return default

    s = s.replace(" ", "")

    # Norsk desimalkomma
    if "," in s and "." not in s:
        s = s.replace(",", ".")

    # Hvis både komma og punktum finnes: anta komma som tusenskiller
    elif "," in s and "." in s:
        s = s.replace(",", "")

    try:
        return float(s)
    except Exception:
        return default


def get_remaining_headers(response):
    return {
        "x-requests-remaining": response.headers.get("x-requests-remaining", ""),
        "x-requests-used": response.headers.get("x-requests-used", ""),
        "x-requests-last": response.headers.get("x-requests-last", "")
    }


def norm_team(name):
    s = str(name).strip()
    aliases = {
        "Man City": "Manchester City",
        "Man United": "Manchester United",
        "Man Utd": "Manchester United",
        "Spurs": "Tottenham Hotspur",
        "Tottenham": "Tottenham Hotspur",
        "Brighton": "Brighton and Hove Albion",
        "Newcastle": "Newcastle United",
        "Nott'm Forest": "Nottingham Forest",
        "Nottm Forest": "Nottingham Forest",
        "Ipswich": "Ipswich Town",
        "Leeds": "Leeds United",
        "Coventry": "Coventry City",
        "Hull": "Hull City",
    }
    return aliases.get(s, s)


# -----------------------
# DEL 1: Hent OddsAPI-data
# -----------------------

def fetch_oddsapi_events():
    if not API_KEY.strip() or API_KEY == "LIM_INN_DIN_API_KEY_HER":
        raise ValueError('API_KEY mangler. Lim inn nøkkelen din i API_KEY = "..."')

    url = f"{BASE_URL}/sports/{SPORT_KEY}/odds"
    params = {
        "apiKey": API_KEY,
        "regions": REGIONS,
        "markets": MARKETS,
        "oddsFormat": ODDS_FORMAT,
        "dateFormat": DATE_FORMAT,
    }

    print("Henter odds fra The Odds API...")
    response = requests.get(url, params=params, timeout=30)

    print("Status code:", response.status_code)
    print("Quota:", get_remaining_headers(response))

    if response.status_code != 200:
        raise RuntimeError(f"OddsAPI-feil {response.status_code}: {response.text[:1000]}")

    events = response.json()
    print(f"Antall events returnert fra OddsAPI: {len(events)}")

    return events


def flatten_oddsapi_events(events):
    raw_rows = []
    market_check_rows = []
    detail_rows = []

    for event in events:
        match_id = event.get("id", "")
        sport_key = event.get("sport_key", "")
        sport_title = event.get("sport_title", "")
        commence_time = event.get("commence_time", "")
        home_team = norm_team(event.get("home_team", ""))
        away_team = norm_team(event.get("away_team", ""))
        bookmakers = event.get("bookmakers", [])

        raw_rows.append({
            "match_id": match_id,
            "sport_key": sport_key,
            "sport_title": sport_title,
            "commence_time": commence_time,
            "home_team": home_team,
            "away_team": away_team,
            "bookmakers_count": len(bookmakers),
            "bookmakers": ", ".join([b.get("title", "") for b in bookmakers])
        })

        h2h_home_prices, h2h_draw_prices, h2h_away_prices = [], [], []
        totals_over_prices, totals_under_prices, totals_points = [], [], []
        markets_found = set()

        for bookmaker in bookmakers:
            bm_key = bookmaker.get("key", "")
            bm_title = bookmaker.get("title", "")
            bm_last_update = bookmaker.get("last_update", "")

            for market in bookmaker.get("markets", []):
                market_key = market.get("key", "")
                markets_found.add(market_key)
                market_last_update = market.get("last_update", "")

                for outcome in market.get("outcomes", []):
                    outcome_name = outcome.get("name", "")
                    price = outcome.get("price", "")
                    point = outcome.get("point", "")
                    description = outcome.get("description", "")

                    detail_rows.append({
                        "match_id": match_id,
                        "sport_key": sport_key,
                        "sport_title": sport_title,
                        "commence_time": commence_time,
                        "home_team": home_team,
                        "away_team": away_team,
                        "bookmaker_key": bm_key,
                        "bookmaker": bm_title,
                        "bookmaker_last_update": bm_last_update,
                        "market": market_key,
                        "market_last_update": market_last_update,
                        "outcome_name": outcome_name,
                        "price": price,
                        "point": point,
                        "description": description,
                    })

                    p = parse_float(price)
                    if pd.isna(p):
                        continue

                    if market_key == "h2h":
                        if outcome_name == home_team:
                            h2h_home_prices.append(p)
                        elif outcome_name == away_team:
                            h2h_away_prices.append(p)
                        elif str(outcome_name).lower() == "draw":
                            h2h_draw_prices.append(p)

                    elif market_key == "totals":
                        if point not in [None, ""]:
                            totals_points.append(point)
                        if str(outcome_name).lower() == "over":
                            totals_over_prices.append(p)
                        elif str(outcome_name).lower() == "under":
                            totals_under_prices.append(p)

        def avg(xs):
            return round(sum(xs) / len(xs), 3) if xs else ""

        def unique_points(xs):
            clean = sorted(list(set([x for x in xs if x not in [None, ""]])))
            return ", ".join([str(x) for x in clean])

        market_check_rows.append({
            "match_id": match_id,
            "commence_time": commence_time,
            "home_team": home_team,
            "away_team": away_team,
            "bookmakers_count": len(bookmakers),
            "markets_found": ", ".join(sorted(markets_found)),
            "avg_home_odds": avg(h2h_home_prices),
            "avg_draw_odds": avg(h2h_draw_prices),
            "avg_away_odds": avg(h2h_away_prices),
            "avg_over_odds": avg(totals_over_prices),
            "avg_under_odds": avg(totals_under_prices),
            "totals_lines_found": unique_points(totals_points),
            "h2h_books": len(h2h_home_prices),
            "totals_books": len(totals_over_prices),
        })

    raw_df = pd.DataFrame(raw_rows)
    market_check_df = pd.DataFrame(market_check_rows)
    detail_df = pd.DataFrame(detail_rows)

    if not raw_df.empty:
        raw_df = raw_df.sort_values(["commence_time", "home_team"]).reset_index(drop=True)

    if not market_check_df.empty:
        market_check_df = market_check_df.sort_values(["commence_time", "home_team"]).reset_index(drop=True)

    if not detail_df.empty:
        detail_df = detail_df.sort_values(
            ["commence_time", "home_team", "bookmaker", "market", "outcome_name"]
        ).reset_index(drop=True)

    return raw_df, market_check_df, detail_df


# -----------------------
# DEL 2: Odds -> kamp-xG
# -----------------------

def avg_no_vig_probs(prices):
    implied = {}

    for outcome, odds_list in prices.items():
        clean = [parse_float(x) for x in odds_list]
        clean = [x for x in clean if not pd.isna(x) and x > 1.0]

        if clean:
            avg_odds = sum(clean) / len(clean)
            implied[outcome] = 1.0 / avg_odds

    total = sum(implied.values())

    if total <= 0:
        return {}

    return {k: v / total for k, v in implied.items()}


def poisson_pmf(lam, k):
    return math.exp(-lam) * (lam ** k) / math.factorial(k)


def poisson_match_probs(home_xg, away_xg, max_goals=10):
    home_win = 0.0
    draw = 0.0
    away_win = 0.0
    total_under_25 = 0.0

    home_zero = poisson_pmf(home_xg, 0)
    away_zero = poisson_pmf(away_xg, 0)

    home_cs = away_zero
    away_cs = home_zero

    for h in range(max_goals + 1):
        ph = poisson_pmf(home_xg, h)

        for a in range(max_goals + 1):
            pa = poisson_pmf(away_xg, a)
            p = ph * pa

            if h > a:
                home_win += p
            elif h == a:
                draw += p
            else:
                away_win += p

            if h + a <= 2:
                total_under_25 += p

    return {
        "home_win": home_win,
        "draw": draw,
        "away_win": away_win,
        "home_cs": home_cs,
        "away_cs": away_cs,
        "under_25": total_under_25,
        "over_25": 1.0 - total_under_25,
    }


def expected_total_from_over25(over_prob):
    if over_prob is None or pd.isna(over_prob):
        return DEFAULT_TOTAL_GOALS

    target = max(0.05, min(0.95, float(over_prob)))

    best_lam = DEFAULT_TOTAL_GOALS
    best_err = 999

    for lam in np.arange(0.60, 5.51, 0.01):
        p_under_25 = sum(poisson_pmf(lam, k) for k in range(0, 3))
        p_over_25 = 1.0 - p_under_25
        err = abs(p_over_25 - target)

        if err < best_err:
            best_err = err
            best_lam = lam

    return float(best_lam)


def fit_xg_from_market(home_win_p, draw_p, away_win_p, total_goals):
    total_goals = max(0.6, min(5.5, float(total_goals)))

    best = {
        "home_xg": total_goals / 2,
        "away_xg": total_goals / 2,
        "err": 999,
        "probs": None,
    }

    for home_share in np.arange(0.15, 0.86, 0.0025):
        hxg = total_goals * home_share
        axg = total_goals - hxg

        probs = poisson_match_probs(hxg, axg)

        err = (
            abs(probs["home_win"] - home_win_p) +
            abs(probs["draw"] - draw_p) +
            abs(probs["away_win"] - away_win_p)
        )

        if err < best["err"]:
            best = {
                "home_xg": hxg,
                "away_xg": axg,
                "err": err,
                "probs": probs,
            }

    return best


def build_market_xg(detail_df):
    required = [
        "match_id",
        "commence_time",
        "home_team",
        "away_team",
        "bookmaker",
        "market",
        "outcome_name",
        "price",
    ]

    missing = [c for c in required if c not in detail_df.columns]
    if missing:
        raise ValueError(f"Input mangler kolonner: {missing}")

    if "point" not in detail_df.columns:
        detail_df["point"] = ""

    rows = []
    check_rows = []

    grouped = detail_df.groupby(["match_id", "commence_time", "home_team", "away_team"], dropna=False)

    for (match_id, kickoff, home, away), g in grouped:
        home = str(home).strip()
        away = str(away).strip()

        h2h = g[g["market"].astype(str).str.lower().eq("h2h")].copy()
        totals = g[g["market"].astype(str).str.lower().eq("totals")].copy()

        h2h_prices = {
            "home": [],
            "draw": [],
            "away": [],
        }

        for _, r in h2h.iterrows():
            outcome = str(r.get("outcome_name", "")).strip()
            price = r.get("price", "")

            if outcome == home:
                h2h_prices["home"].append(price)
            elif outcome == away:
                h2h_prices["away"].append(price)
            elif outcome.lower() == "draw":
                h2h_prices["draw"].append(price)

        h2h_probs = avg_no_vig_probs(h2h_prices)

        home_p = h2h_probs.get("home", np.nan)
        draw_p = h2h_probs.get("draw", np.nan)
        away_p = h2h_probs.get("away", np.nan)

        over_prices = []
        under_prices = []
        selected_point = np.nan

        if not totals.empty:
            totals["point_num"] = totals["point"].apply(parse_float)
            points = sorted([p for p in totals["point_num"].dropna().unique().tolist()])

            if points:
                selected_point = min(points, key=lambda x: abs(x - 2.5))
                tsel = totals[totals["point_num"].eq(selected_point)]

                for _, r in tsel.iterrows():
                    outcome = str(r.get("outcome_name", "")).strip().lower()
                    price = r.get("price", "")

                    if outcome == "over":
                        over_prices.append(price)
                    elif outcome == "under":
                        under_prices.append(price)

        total_probs = avg_no_vig_probs({
            "over": over_prices,
            "under": under_prices,
        })

        over_p = total_probs.get("over", np.nan)
        total_goals = expected_total_from_over25(over_p)

        if any(pd.isna(x) for x in [home_p, draw_p, away_p]):
            check_rows.append({
                "match_id": match_id,
                "home_team": home,
                "away_team": away,
                "status": "MISSING_H2H",
                "h2h_rows": len(h2h),
                "totals_rows": len(totals),
            })
            continue

        fit = fit_xg_from_market(home_p, draw_p, away_p, total_goals)
        hxg = fit["home_xg"]
        axg = fit["away_xg"]
        probs = fit["probs"] or poisson_match_probs(hxg, axg)

        rows.append({
            "Kickoff": kickoff,
            "Team": home,
            "Opponent": away,
            "H/A": "H",
            "Team xG": round(hxg, 4),
            "Opp xG": round(axg, 4),
            "CS%": round(probs["home_cs"] * 100, 4),
            "Win%": round(probs["home_win"] * 100, 4),
            "Draw%": round(probs["draw"] * 100, 4),
            "Loss%": round(probs["away_win"] * 100, 4),
            "Total xG": round(hxg + axg, 4),
            "Fit Error": round(fit["err"], 4),
            "Match ID": match_id,
        })

        rows.append({
            "Kickoff": kickoff,
            "Team": away,
            "Opponent": home,
            "H/A": "A",
            "Team xG": round(axg, 4),
            "Opp xG": round(hxg, 4),
            "CS%": round(probs["away_cs"] * 100, 4),
            "Win%": round(probs["away_win"] * 100, 4),
            "Draw%": round(probs["draw"] * 100, 4),
            "Loss%": round(probs["home_win"] * 100, 4),
            "Total xG": round(hxg + axg, 4),
            "Fit Error": round(fit["err"], 4),
            "Match ID": match_id,
        })

        check_rows.append({
            "match_id": match_id,
            "kickoff": kickoff,
            "home_team": home,
            "away_team": away,
            "status": "OK",
            "h2h_home_p": round(home_p, 4),
            "h2h_draw_p": round(draw_p, 4),
            "h2h_away_p": round(away_p, 4),
            "selected_total_line": selected_point,
            "over_p": round(over_p, 4) if not pd.isna(over_p) else "",
            "total_goals_est": round(total_goals, 4),
            "home_xg": round(hxg, 4),
            "away_xg": round(axg, 4),
            "fit_error": round(fit["err"], 4),
            "h2h_rows": len(h2h),
            "totals_over_rows": len(over_prices),
            "totals_under_rows": len(under_prices),
        })

    out_df = pd.DataFrame(rows)
    check_df = pd.DataFrame(check_rows)

    if not out_df.empty:
        out_df = out_df.sort_values(["Kickoff", "Team"]).reset_index(drop=True)

    if not check_df.empty:
        check_df = check_df.sort_values(["kickoff", "home_team"], na_position="last").reset_index(drop=True)

    return out_df, check_df


# -----------------------
# DEL 3: kamp-xG -> Team Strength / Match Strength
# -----------------------

def scale_10_95(series):
    s = pd.to_numeric(series, errors="coerce")
    mn = s.min()
    mx = s.max()

    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series([50.0] * len(series), index=series.index)

    return (MIN_SCALE + (s - mn) / (mx - mn) * (MAX_SCALE - MIN_SCALE)).clip(MIN_SCALE, MAX_SCALE)


def build_team_and_match_strength(market_xg_df):
    df = market_xg_df.copy()

    required = [
        "Kickoff", "Team", "Opponent", "H/A",
        "Team xG", "Opp xG", "CS%", "Win%", "Draw%", "Loss%"
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Mangler kolonner i Market_xG_All_Fixtures_v1: {missing}")

    check_df = df[[
        "Kickoff", "Team", "Opponent", "H/A",
        "Team xG", "Opp xG", "CS%", "Win%", "Draw%", "Loss%"
    ]].copy()

    check_df = check_df.rename(columns={
        "Team xG": "Raw Team xG",
        "Opp xG": "Raw Opp xG",
        "CS%": "Raw CS%",
        "Win%": "Raw Win%",
        "Draw%": "Raw Draw%",
        "Loss%": "Raw Loss%"
    })

    for col in ["Team xG", "Opp xG", "CS%", "Win%", "Draw%", "Loss%", "Total xG", "Fit Error"]:
        if col in df.columns:
            df[col] = df[col].apply(parse_float)

    df = df.dropna(subset=["Team xG", "Opp xG"]).copy()

    if df.empty:
        raise ValueError("Ingen gyldige xG-rader funnet i Market_xG_All_Fixtures_v1")

    check_df = check_df.loc[df.index].copy()
    check_df["Parsed Team xG"] = df["Team xG"].values
    check_df["Parsed Opp xG"] = df["Opp xG"].values
    check_df["Check Note"] = ""

    league_avg_xg = df["Team xG"].mean()

    team = (
        df.groupby("Team", as_index=False)
        .agg({
            "Team xG": ["mean", "sum", "count"],
            "Opp xG": ["mean", "sum"],
            "CS%": "mean",
            "Win%": "mean",
            "Draw%": "mean",
            "Loss%": "mean"
        })
    )

    team.columns = [
        "Team",
        "Avg Team xG",
        "Total Team xG",
        "Matches",
        "Avg Opp xG",
        "Total Opp xG",
        "Avg CS%",
        "Avg Win%",
        "Avg Draw%",
        "Avg Loss%"
    ]

    team["League Avg xG"] = league_avg_xg
    team["Team Att Ratio"] = team["Avg Team xG"] / league_avg_xg
    team["Team Def Ratio"] = league_avg_xg / team["Avg Opp xG"]

    team["Team Att"] = scale_10_95(team["Team Att Ratio"])
    team["Team Def"] = scale_10_95(team["Team Def Ratio"])

    team["Interpretation"] = team.apply(
        lambda r: f"Att {r['Team Att Ratio']:.2f}x league avg, Def {r['Team Def Ratio']:.2f}x league avg",
        axis=1
    )

    team_out = team[[
        "Team",
        "Matches",
        "Avg Team xG",
        "Avg Opp xG",
        "League Avg xG",
        "Team Att Ratio",
        "Team Def Ratio",
        "Team Att",
        "Team Def",
        "Avg CS%",
        "Avg Win%",
        "Avg Draw%",
        "Avg Loss%",
        "Total Team xG",
        "Total Opp xG",
        "Interpretation"
    ]].copy()

    team_att_map = dict(zip(team["Team"], team["Team Att"]))
    team_def_map = dict(zip(team["Team"], team["Team Def"]))
    team_att_ratio_map = dict(zip(team["Team"], team["Team Att Ratio"]))
    team_def_ratio_map = dict(zip(team["Team"], team["Team Def Ratio"]))

    match = df.copy()

    match["Team Att"] = match["Team"].map(team_att_map)
    match["Team Def"] = match["Team"].map(team_def_map)
    match["Opp Team Att"] = match["Opponent"].map(team_att_map)
    match["Opp Team Def"] = match["Opponent"].map(team_def_map)

    match["Team Att Ratio"] = match["Team"].map(team_att_ratio_map)
    match["Team Def Ratio"] = match["Team"].map(team_def_ratio_map)
    match["Opp Team Att Ratio"] = match["Opponent"].map(team_att_ratio_map)
    match["Opp Team Def Ratio"] = match["Opponent"].map(team_def_ratio_map)

    match["Match Att"] = match["Team Att"] - match["Opp Team Def"]
    match["Match Def"] = match["Team Def"] - match["Opp Team Att"]

    match["Match Att Ratio Diff"] = match["Team Att Ratio"] - match["Opp Team Def Ratio"]
    match["Match Def Ratio Diff"] = match["Team Def Ratio"] - match["Opp Team Att Ratio"]

    match_out_cols = [
        "Kickoff",
        "Team",
        "Opponent",
        "H/A",
        "Team Att",
        "Opp Team Def",
        "Match Att",
        "Team xG",
        "Team Def",
        "Opp Team Att",
        "Match Def",
        "Opp xG",
        "CS%",
        "Win%",
        "Draw%",
        "Loss%",
        "Team Att Ratio",
        "Opp Team Def Ratio",
        "Match Att Ratio Diff",
        "Team Def Ratio",
        "Opp Team Att Ratio",
        "Match Def Ratio Diff"
    ]

    match_out = match[match_out_cols].copy()

    for out_df in [check_df, team_out, match_out]:
        for col in out_df.columns:
            if pd.api.types.is_numeric_dtype(out_df[col]):
                out_df[col] = out_df[col].round(4)

    team_out = team_out.sort_values("Team Att", ascending=False).reset_index(drop=True)
    match_out = match_out.sort_values(["Kickoff", "Team"]).reset_index(drop=True)

    return check_df, team_out, match_out, league_avg_xg


# -----------------------
# RUN
# -----------------------

events = fetch_oddsapi_events()

raw_df, market_check_df, detail_df = flatten_oddsapi_events(events)

update_worksheet("OddsAPI_Raw_Matches", raw_df, min_rows=1000, min_cols=80)
update_worksheet("OddsAPI_Market_Check", market_check_df, min_rows=1000, min_cols=80)
update_worksheet("OddsAPI_Bookmaker_Detail", detail_df, min_rows=2000, min_cols=80)

market_xg_df, parse_check_df = build_market_xg(detail_df)

update_worksheet("Market_xG_All_Fixtures_v1", market_xg_df, min_rows=2000, min_cols=80)
update_worksheet("Market_xG_Parse_Check_v1", parse_check_df, min_rows=1000, min_cols=80)

decimal_check_df, team_strength_df, match_strength_df, league_avg_xg = build_team_and_match_strength(market_xg_df)

update_worksheet("Market_Decimal_Check_v2_1", decimal_check_df, min_rows=1000, min_cols=80)
update_worksheet("Market_Team_Strength_v2_1", team_strength_df, min_rows=200, min_cols=80)
update_worksheet("Market_Match_Strength_v2_1", match_strength_df, min_rows=2000, min_cols=80)

print("")
print("FERDIG: Jason Market Master v1")
print(f"Antall kamper fra OddsAPI: {len(raw_df)}")
print(f"Market_xG-rader: {len(market_xg_df)}")
print(f"Market_Match_Strength-rader: {len(match_strength_df)}")
print(f"League Avg xG i grunnlaget: {league_avg_xg:.4f}")
print("")
print("Neste steg:")
print("Kjør Jason-hovedscriptet.")

Henter odds fra The Odds API...
Status code: 200
Quota: {'x-requests-remaining': '492', 'x-requests-used': '8', 'x-requests-last': '2'}
Antall events returnert fra OddsAPI: 10
Oppdaterer OddsAPI_Raw_Matches...
Oppdaterer OddsAPI_Market_Check...
Oppdaterer OddsAPI_Bookmaker_Detail...
Oppdaterer Market_xG_All_Fixtures_v1...
Oppdaterer Market_xG_Parse_Check_v1...
Oppdaterer Market_Decimal_Check_v2_1...
Oppdaterer Market_Team_Strength_v2_1...
Oppdaterer Market_Match_Strength_v2_1...

FERDIG: Jason Market Master v1
Antall kamper fra OddsAPI: 10
Market_xG-rader: 20
Market_Match_Strength-rader: 20
League Avg xG i grunnlaget: 1.5430

Neste steg:
Kjør Jason-hovedscriptet.
